<a href="https://colab.research.google.com/github/kufreetok/sto-experiment-analysis/blob/main/01_send_time_optimisation_data_generation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Document Assumptions

In [ ]:
"""
PROJECT:
Evaluating Send-Time Optimisation to Improve Marketing Pipeline Performance

PURPOSE:
Assess whether send-time optimisation increases marketing pipeline generation
compared with fixed-time email delivery.

ASSUMPTIONS:

Population Size: 100,000 Leads

Experiment Split:
- 50% Control
- 50% Treatment

Control:
- Emails sent at 10 AM

Treatment:
- Emails sent at predicted optimal hour

Baseline Funnel:

Open Rate: 35%
CTR: 6%
MQO Rate: 2.0%

Expected Treatment Effects

Technology:
- Open Rate +3.5%
- CTR +12%
- MQO +20%

Finance:
- Open Rate +2.0%
- CTR +8%
- MQO +10%

Healthcare:
- Open Rate +1.0%
- CTR +3%
- MQO +5%

Manufacturing:
- No meaningful lift

Average Opportunity Value:
£15,000
"""

'\nPROJECT:\nEvaluating Send-Time Optimisation to Improve Marketing Pipeline Performance\n\nPURPOSE:\nAssess whether send-time optimisation increases marketing pipeline generation\ncompared with fixed-time email delivery.\n\nASSUMPTIONS:\n\nPopulation Size: 100,000 Leads\n\nExperiment Split:\n- 50% Control\n- 50% Treatment\n\nControl:\n- Emails sent at 10 AM\n\nTreatment:\n- Emails sent at predicted optimal hour\n\nBaseline Funnel:\n\nOpen Rate: 35%\nCTR: 6%\nMQO Rate: 2.0%\n\nExpected Treatment Effects\n\nTechnology:\n- Open Rate +3.5%\n- CTR +12%\n- MQO +20%\n\nFinance:\n- Open Rate +2.0%\n- CTR +8%\n- MQO +10%\n\nHealthcare:\n- Open Rate +1.0%\n- CTR +3%\n- MQO +5%\n\nManufacturing:\n- No meaningful lift\n\nAverage Opportunity Value:\n£15,000\n'

In [ ]:
import pandas as pd
import numpy as np

np.random.seed(42)


In [ ]:
n_leads = 50000

lead_ids = np.arange(1, n_leads + 1)

industries = np.random.choice(
    ["Technology", "Finance", "Manufacturing", "Healthcare"],
    size=n_leads,
    p=[0.35, 0.25, 0.20, 0.20]
)

company_sizes = np.random.choice(
    ["SMB", "Mid-Market", "Enterprise"],
    size=n_leads,
    p=[0.50, 0.35, 0.15]
)

regions = np.random.choice(
    ["UK", "US", "EMEA"],
    size=n_leads,
    p=[0.30, 0.50, 0.20]
)

lead_sources = np.random.choice(
    ["Organic", "Paid Search", "Webinar", "Referral"],
    size=n_leads,
    p=[0.30, 0.30, 0.25, 0.15]
)

experiment_group = np.random.choice(
    ["Control", "Treatment"],
    size=n_leads,
    p=[0.50, 0.50]
)

leads = pd.DataFrame({
    "lead_id": lead_ids,
    "industry": industries,
    "company_size": company_sizes,
    "region": regions,
    "lead_source": lead_sources,
    "experiment_group": experiment_group
})

leads.head()

,lead_id,industry,company_size,region,lead_source,experiment_group
0,1,Finance,Mid-Market,US,Organic,Control
1,2,Healthcare,SMB,US,Paid Search,Control
2,3,Manufacturing,SMB,US,Webinar,Control
3,4,Finance,Mid-Market,US,Webinar,Control
4,5,Technology,SMB,US,Webinar,Control


In [ ]:
leads["experiment_group"].value_counts()

,count
experiment_group,
Control,25007
Treatment,24993


In [ ]:
pd.crosstab(
    leads["industry"],
    leads["experiment_group"]
)


experiment_group,Control,Treatment
industry,,
Finance,6220,6221
Healthcare,5092,4834
Manufacturing,5028,4997
Technology,8667,8941


In [ ]:
leads.to_csv("leads.csv", index=False)

In [ ]:
# Create email IDs

email_events = leads.copy()

email_events["email_id"] = range(
    1,
    len(email_events) + 1
)

In [ ]:
possible_hours = [8, 9, 10, 11, 13, 14, 15, 16]

In [ ]:
email_events.loc[:, "send_hour"] = np.where(
    email_events["experiment_group"] == "Control",
    10,
    np.random.choice(
        possible_hours,
        size=len(email_events),
        p=[
            0.10,
            0.15,
            0.20,
            0.15,
            0.10,
            0.10,
            0.10,
            0.10
        ]
    )
)

In [ ]:
email_events.loc[:, "delivered"] = np.random.binomial(
    1,
    0.98,
    len(email_events)
)

In [ ]:
email_events = email_events[
    [
        "email_id",
        "lead_id",
        "experiment_group",
        "send_hour",
        "delivered"
    ]
]

In [ ]:
email_events.shape

(50000, 5)

In [ ]:
email_events["experiment_group"].value_counts()

,count
experiment_group,
Control,25007
Treatment,24993


In [ ]:
email_events.groupby(
    "experiment_group"
)["send_hour"].value_counts()

experiment_group  send_hour
Control           10           25007
Treatment         10            4995
                  11            3714
                  9             3699
                  13            2606
                  14            2548
                  15            2491
                  8             2480
                  16            2460
Name: count, dtype: int64

In [ ]:
email_events = leads.copy(deep=True)

email_events["email_id"] = range(
    1,
    len(email_events) + 1
)

possible_hours = [8, 9, 10, 11, 13, 14, 15, 16]

email_events["send_hour"] = np.where(
    email_events["experiment_group"] == "Control",
    10,
    np.random.choice(
        possible_hours,
        size=len(email_events),
        p=[0.10, 0.15, 0.20, 0.15, 0.10, 0.10, 0.10, 0.10]
    )
)

email_events["delivered"] = np.random.binomial(
    1,
    0.98,
    len(email_events)
)

email_events = email_events[
    [
        "email_id",
        "lead_id",
        "experiment_group",
        "send_hour",
        "delivered"
    ]
].copy()

In [ ]:
email_events.head()


,email_id,lead_id,experiment_group,send_hour,delivered
0,1,1,Control,10,1
1,2,2,Control,10,1
2,3,3,Control,10,1
3,4,4,Control,10,1
4,5,5,Control,10,1


In [ ]:
email_events.shape

(50000, 5)

In [ ]:
email_events.describe(include="all")


,email_id,lead_id,experiment_group,send_hour,delivered
count,50000.000000,50000.000000,50000,50000.000000,50000.000000
unique,NaN,NaN,2,NaN,NaN
top,NaN,NaN,Control,NaN,NaN
freq,NaN,NaN,25007,NaN,NaN
mean,25000.500000,25000.500000,NaN,10.798340,0.979660
std,14433.901067,14433.901067,NaN,2.003785,0.141162
min,1.000000,1.000000,NaN,8.000000,0.000000
25%,12500.750000,12500.750000,NaN,10.000000,1.000000
50%,25000.500000,25000.500000,NaN,10.000000,1.000000
75%,37500.250000,37500.250000,NaN,11.000000,1.000000


In [ ]:
email_events.groupby(
    "experiment_group"
)["send_hour"].value_counts().sort_index()

experiment_group  send_hour
Control           10           25007
Treatment         8             2580
                  9             3611
                  10            5079
                  11            3757
                  13            2440
                  14            2516
                  15            2513
                  16            2497
Name: count, dtype: int64

In [ ]:
%whos


Variable               Type         Data/Info
---------------------------------------------
base_mqo               dict         n=4
company_sizes          ndarray      50000: 50000 elems, type `<U10`, 2000000 bytes (1.9073486328125 Mb)
control_ctr            float        0.06
control_open           dict         n=4
datetime               type         <class 'datetime.datetime'>
deal_value             float        9910.503923512268
duckdb                 module       <module 'duckdb' from '/u<...>ages/duckdb/__init__.py'>
email_events           DataFrame           email_id  lead_id <...>n[50000 rows x 5 columns]
engagement_base        DataFrame           email_id  lead_id <...>[50000 rows x 19 columns]
engagement_events      DataFrame           email_id  opened  <...>n[50000 rows x 5 columns]
experiment_group       ndarray      50000: 50000 elems, type `<U9`, 1800000 bytes (1.71661376953125 Mb)
get_open_prob          function     <function get_open_prob at 0x7d291c4a7060>
get_opportunit

In [ ]:
from datetime import datetime, timedelta

In [ ]:
start_date = datetime(2025, 1, 1)

In [ ]:
email_events["send_date"] = (
    start_date +
    pd.to_timedelta(
        np.random.randint(
            0,
            56,
            size=len(email_events)
        ),
        unit="D"
    )
)

In [ ]:
email_events["email_type"] = np.random.choice(
    [
        "Newsletter",
        "Webinar",
        "Product Update"
    ],
    size=len(email_events),
    p=[0.5, 0.3, 0.2]
)

In [ ]:
email_events["campaign_id"] = np.random.choice(
    [
        "CMP001",
        "CMP002",
        "CMP003",
        "CMP004",
        "CMP005"
    ],
    size=len(email_events)
)

In [ ]:
email_events = email_events[
    [
        "email_id",
        "lead_id",
        "campaign_id",
        "email_type",
        "experiment_group",
        "send_date",
        "send_hour",
        "delivered"
    ]
].copy()

In [ ]:
engagement_base = (
    email_events.merge(
        leads[
            [
                "lead_id",
                "industry"
            ]
        ],
        on="lead_id",
        how="left"
    )
)

engagement_base.head()

,email_id,lead_id,campaign_id,email_type,experiment_group,send_date,send_hour,delivered,industry
0,1,1,CMP002,Newsletter,Control,2025-01-16,10,1,Finance
1,2,2,CMP002,Product Update,Control,2025-01-05,10,1,Healthcare
2,3,3,CMP005,Product Update,Control,2025-02-02,10,1,Manufacturing
3,4,4,CMP004,Newsletter,Control,2025-01-19,10,1,Finance
4,5,5,CMP004,Product Update,Control,2025-02-16,10,1,Technology


In [ ]:
email_events["send_datetime"] = (
    email_events["send_date"]
    + pd.to_timedelta(
        email_events["send_hour"],
        unit="h"
    )
)

In [ ]:
email_events = email_events[
    [
        "email_id",
        "lead_id",
        "experiment_group",
        "send_date",
        "send_datetime",
        "send_hour",
        "delivered"
    ]
].copy()

In [ ]:
email_events.head(10)


,email_id,lead_id,experiment_group,send_date,send_datetime,send_hour,delivered
0,1,1,Control,2025-01-16,2025-01-16 10:00:00,10,1
1,2,2,Control,2025-01-05,2025-01-05 10:00:00,10,1
2,3,3,Control,2025-02-02,2025-02-02 10:00:00,10,1
3,4,4,Control,2025-01-19,2025-01-19 10:00:00,10,1
4,5,5,Control,2025-02-16,2025-02-16 10:00:00,10,1
5,6,6,Control,2025-02-16,2025-02-16 10:00:00,10,1
6,7,7,Control,2025-02-18,2025-02-18 10:00:00,10,1
7,8,8,Treatment,2025-01-25,2025-01-25 11:00:00,11,1
8,9,9,Treatment,2025-02-25,2025-02-25 11:00:00,11,1
9,10,10,Control,2025-02-11,2025-02-11 10:00:00,10,1


In [ ]:
email_events.dtypes

,0
email_id,int64
lead_id,int64
experiment_group,object
send_date,datetime64[ns]
send_datetime,datetime64[ns]
send_hour,int64
delivered,int64


In [ ]:
email_events["delivered"] = email_events["delivered"].astype(bool)

In [ ]:
email_events["lead_id"].nunique()

50000

In [ ]:
leads["lead_id"].nunique()

50000

In [ ]:
engagement_base = email_events.merge(
    leads[["lead_id", "industry", "company_size", "region"]],
    on="lead_id",
    how="left"
)

engagement_base.head()

,email_id,lead_id,experiment_group,send_date,send_datetime,send_hour,delivered,industry,company_size,region
0,1,1,Control,2025-01-16,2025-01-16 10:00:00,10,True,Finance,Mid-Market,US
1,2,2,Control,2025-01-05,2025-01-05 10:00:00,10,True,Healthcare,SMB,US
2,3,3,Control,2025-02-02,2025-02-02 10:00:00,10,True,Manufacturing,SMB,US
3,4,4,Control,2025-01-19,2025-01-19 10:00:00,10,True,Finance,Mid-Market,US
4,5,5,Control,2025-02-16,2025-02-16 10:00:00,10,True,Technology,SMB,US


In [ ]:
control_open = {
    "Technology": 0.35,
    "Finance": 0.34,
    "Healthcare": 0.33,
    "Manufacturing": 0.32
}

In [ ]:
treatment_lift = {
    "Technology": 0.035,
    "Finance": 0.02,
    "Healthcare": 0.01,
    "Manufacturing": 0.00
}

In [ ]:
def get_open_prob(row):

    base = control_open[row["industry"]]

    if row["experiment_group"] == "Treatment":
        base += treatment_lift[row["industry"]]

    return base

In [ ]:
engagement_base["open_prob"] = engagement_base.apply(
    get_open_prob,
    axis=1
)

In [ ]:
engagement_base["opened"] = np.random.binomial(
    1,
    engagement_base["open_prob"]
)

In [ ]:
control_ctr = 0.06
treatment_ctr = 0.066

In [ ]:
engagement_base["clicked"] = np.where(
    engagement_base["opened"] == 1,
    np.random.binomial(
        1,
        np.where(
            engagement_base["experiment_group"] == "Treatment",
            treatment_ctr,
            control_ctr
        )
    ),
    0
)

In [ ]:
engagement_base["unsubscribed"] = np.where(
    engagement_base["experiment_group"] == "Treatment",
    np.random.binomial(1, 0.0042, len(engagement_base)),
    np.random.binomial(1, 0.0035, len(engagement_base))
)

In [ ]:
engagement_base["complaint"] = np.random.binomial(
    1,
    0.0005,
    len(engagement_base)
)

In [ ]:
engagement_events = engagement_base[
    [
        "email_id",
        "opened",
        "clicked",
        "unsubscribed",
        "complaint"
    ]
].copy()

In [ ]:
engagement_base["bot_open"] = np.random.binomial(
    1,
    0.03,  # 3% bot activity
    len(engagement_base)
)

In [ ]:
engagement_base["tracking_error"] = np.random.binomial(
    1,
    0.015,  # 1.5% logging issues
    len(engagement_base)
)

In [ ]:
engagement_base["true_opened"] = np.random.binomial(
    1,
    engagement_base["open_prob"]
)

In [ ]:
engagement_base["opened"] = np.where(
    engagement_base["true_opened"] == 1,
    1,
    engagement_base["bot_open"]
)

In [ ]:
engagement_base[
    (engagement_base["clicked"] == 1) &
    (engagement_base["opened"].isna())
].shape[0]

0

In [ ]:
engagement_base.loc[
    engagement_base["tracking_error"] == 1,
    "opened"
] = np.nan

In [ ]:
control_ctr = 0.06
treatment_ctr = 0.066

In [ ]:
engagement_base["clicked"] = np.where(
    engagement_base["experiment_group"] == "Treatment",
    np.random.binomial(1, treatment_ctr, len(engagement_base)),
    np.random.binomial(1, control_ctr, len(engagement_base))
)

In [ ]:
engagement_base.loc[
    (engagement_base["opened"].isna()) &
    (engagement_base["clicked"] == 1),
    "clicked"
] = 1

In [ ]:
engagement_base["unsubscribed"] = np.where(
    engagement_base["experiment_group"] == "Treatment",
    np.random.binomial(1, 0.0042, len(engagement_base)),
    np.random.binomial(1, 0.0035, len(engagement_base))
)

In [ ]:
engagement_base["complaint"] = np.random.binomial(
    1,
    0.0005,
    len(engagement_base)
)

In [ ]:
engagement_events = engagement_base[
    [
        "email_id",
        "opened",
        "clicked",
        "unsubscribed",
        "complaint"
    ]
].copy()

In [ ]:
engagement_events["opened"].isna().mean()

np.float64(0.01538)

In [ ]:
engagement_events["clicked"].mean()

np.float64(0.06386)

In [ ]:
op_base = engagement_base.copy()

In [ ]:
base_mqo = {
    "Technology": 0.025,
    "Finance": 0.020,
    "Healthcare": 0.015,
    "Manufacturing": 0.010
}

In [ ]:
def get_opportunity_prob(row):

    base = base_mqo[row["industry"]]

    if row["experiment_group"] == "Treatment":
        base += treatment_lift[row["industry"]]

    # engagement multipliers
    if row["opened"] == 1:
        base *= 1.5

    if row["clicked"] == 1:
        base *= 2.5

    return base

In [ ]:
op_base["opportunity_prob"] = op_base.apply(
    get_opportunity_prob,
    axis=1
)

In [ ]:
op_base["opportunity_created"] = np.random.binomial(
    1,
    op_base["opportunity_prob"]
)

In [ ]:
opportunities = op_base[
    op_base["opportunity_created"] == 1
][
    [
        "email_id",
        "lead_id",
        "industry",
        "experiment_group",
        "opportunity_created"
    ]
].copy()

In [ ]:
opportunities["deal_value"] = np.random.lognormal(
    mean=9,
    sigma=1,
    size=len(opportunities)
)

This code simulates the potential `deal_value` for each identified opportunity. It uses a statistical distribution, specifically a log-normal distribution, to mimic how deal sizes typically vary in real-world sales, with some deals being smaller and a few being very large. This simulation helps in creating a realistic range of potential revenue generated from the marketing efforts. It provides a foundational estimate of the financial impact of each opportunity before considering whether it closes.

In [ ]:
opportunities["deal_value"] = opportunities["deal_value"].round(0)

In [ ]:
opportunities["deal_value"] = opportunities["deal_value"].round(0)

After simulating the raw deal values, this step rounds those values to the nearest whole number. This is a practical step to ensure that deal values are represented in a clean, easily understandable format, typically in whole currency units. It makes the simulated financial data more presentable and comparable to actual sales figures. This ensures consistency and simplifies further analysis of the revenue potential.

In [ ]:
opportunities["closed_won"] = np.random.binomial(
    1,
    0.25,  # 25% win rate typical SMB-mid SaaS blend
    len(opportunities)
)

This line simulates whether each opportunity ultimately results in a 'closed-won' deal. It uses a 25% win rate, which is a common conversion benchmark in B2B sales for SMB to mid-market SaaS companies. This step is crucial for estimating the actual revenue generated from the marketing leads, as not all opportunities turn into successful sales. It allows us to calculate the true value of the pipeline, considering sales conversion probabilities.

In [ ]:
opportunities = opportunities[
    [
        "email_id",
        "lead_id",
        "industry",
        "experiment_group",
        "deal_value",
        "closed_won"
    ]
].copy()

Here, we are finalizing the `opportunities` dataset by selecting only the most relevant columns for analysis. This includes unique identifiers for emails and leads, the industry, the experiment group, the simulated `deal_value`, and whether the deal was `closed_won`. By streamlining the data, we create a clean and focused dataset ready for further analysis and reporting. This ensures that only the critical information needed to evaluate the experiment's financial success is retained.

In [ ]:
opportunities.shape

(1858, 6)

This command simply shows the number of rows and columns in our `opportunities` dataset. The first number (1858 in this case) tells us how many distinct opportunities were generated and identified from the initial pool of marketing leads. The second number (6) indicates how many pieces of information (columns) we have for each of these opportunities. This gives us a quick overview of the scale of the generated sales pipeline.

In [ ]:
opportunities["deal_value"].sum()

np.float64(26020459.0)

Finally, this calculates the sum of all `deal_value` entries within the `opportunities` dataset. This figure represents the total potential revenue if every generated opportunity were to close, providing a high-level view of the marketing pipeline's worth. It's a key metric for understanding the overall financial impact of the simulated marketing activities. This aggregate value is essential for comparing the revenue generation performance across different experiment groups.

In [ ]:
# Load tables

leads
email_events
engagement_events
opportunities

# 1. Experiment validation

# balance checks
# delivery checks

# 2. KPI creation

# open rate
# CTR
# unsubscribe rate

# 3. Funnel creation

# lead level dataset

# 4. Treatment vs control comparison

# 5. Segment analysis

# 6. Incremental pipeline/revenue

# 7. Recommendation


,email_id,lead_id,industry,experiment_group,deal_value,closed_won
89,90,90,Finance,Treatment,18081.0,1
174,175,175,Manufacturing,Treatment,8024.0,0
177,178,178,Finance,Treatment,2620.0,0
190,191,191,Technology,Treatment,847.0,0
297,298,298,Technology,Control,24073.0,0
...,...,...,...,...,...,...
49927,49928,49928,Technology,Treatment,6730.0,1
49929,49930,49930,Technology,Treatment,6983.0,0
49950,49951,49951,Technology,Treatment,1793.0,1
49963,49964,49964,Healthcare,Treatment,9351.0,1


In [ ]:
%pip install duckdb

In [ ]:
import duckdb

duckdb.sql("""
SELECT
    experiment_group,
    COUNT(*) AS emails
FROM email_events
GROUP BY experiment_group
""").df()

,experiment_group,emails
0,Treatment,24993
1,Control,25007


In [ ]:
import duckdb

def q(sql):
    return duckdb.sql(sql).df()


In [ ]:
q("""
SELECT
    experiment_group,
    COUNT(*) AS email_count,
    ROUND(
        COUNT(*) * 100.0 /
        SUM(COUNT(*)) OVER (),
        2
    ) AS pct_of_total
FROM email_events
GROUP BY experiment_group
""")

,experiment_group,email_count,pct_of_total
0,Treatment,24993,49.99
1,Control,25007,50.01


In [ ]:
q("""
SELECT
    experiment_group,
    COUNT(DISTINCT lead_id) AS leads
FROM email_events
GROUP BY experiment_group
""")

,experiment_group,leads
0,Treatment,24993
1,Control,25007


In [ ]:
q("""
WITH assignments AS (

SELECT
    lead_id,
    COUNT(DISTINCT experiment_group) AS groups_seen
FROM email_events
GROUP BY lead_id

)

SELECT
    groups_seen,
    COUNT(*) AS leads
FROM assignments
GROUP BY groups_seen
ORDER BY groups_seen
""")

,groups_seen,leads
0,1,50000


In [ ]:
q("""
SELECT
    experiment_group,
    industry,
    COUNT(*) AS leads
FROM (

    SELECT DISTINCT
        l.lead_id,
        l.industry,
        e.experiment_group

    FROM leads l
    JOIN email_events e
        ON l.lead_id = e.lead_id

)

GROUP BY
    experiment_group,
    industry

ORDER BY
    industry,
    experiment_group
""")

,experiment_group,industry,leads
0,Control,Finance,6220
1,Treatment,Finance,6221
2,Control,Healthcare,5092
3,Treatment,Healthcare,4834
4,Control,Manufacturing,5028
5,Treatment,Manufacturing,4997
6,Control,Technology,8667
7,Treatment,Technology,8941


In [ ]:
q("""
SELECT
    experiment_group,
    region,
    COUNT(*) AS leads
FROM (

    SELECT DISTINCT
        l.lead_id,
        l.region,
        e.experiment_group

    FROM leads l
    JOIN email_events e
        ON l.lead_id = e.lead_id

)

GROUP BY
    experiment_group,
    region

ORDER BY
    region,
    experiment_group
""")

,experiment_group,region,leads
0,Control,EMEA,5043
1,Treatment,EMEA,5116
2,Control,UK,7417
3,Treatment,UK,7417
4,Control,US,12547
5,Treatment,US,12460


In [ ]:
q("""
SELECT
    experiment_group,
    company_size,
    COUNT(*) AS leads
FROM (

    SELECT DISTINCT
        l.lead_id,
        l.company_size,
        e.experiment_group

    FROM leads l
    JOIN email_events e
        ON l.lead_id = e.lead_id

)

GROUP BY
    experiment_group,
    company_size

ORDER BY
    company_size,
    experiment_group
""")

,experiment_group,company_size,leads
0,Control,Enterprise,3769
1,Treatment,Enterprise,3661
2,Control,Mid-Market,8907
3,Treatment,Mid-Market,8782
4,Control,SMB,12331
5,Treatment,SMB,12550


In [ ]:
email_events.columns.tolist()

['email_id',
 'lead_id',
 'experiment_group',
 'send_date',
 'send_datetime',
 'send_hour',
 'delivered']

In [ ]:
email_events.head()

,email_id,lead_id,experiment_group,send_date,send_datetime,send_hour,delivered
0,1,1,Control,2025-01-16,2025-01-16 10:00:00,10,True
1,2,2,Control,2025-01-05,2025-01-05 10:00:00,10,True
2,3,3,Control,2025-02-02,2025-02-02 10:00:00,10,True
3,4,4,Control,2025-01-19,2025-01-19 10:00:00,10,True
4,5,5,Control,2025-02-16,2025-02-16 10:00:00,10,True


In [ ]:
print(leads.columns.tolist())
print(email_events.columns.tolist())
print(engagement_events.columns.tolist())
print(opportunities.columns.tolist())

['lead_id', 'industry', 'company_size', 'region', 'lead_source', 'experiment_group']
['email_id', 'lead_id', 'experiment_group', 'send_date', 'send_datetime', 'send_hour', 'delivered']
['email_id', 'opened', 'clicked', 'unsubscribed', 'complaint']
['email_id', 'lead_id', 'industry', 'experiment_group', 'deal_value', 'closed_won']


In [ ]:
import duckdb

def q(sql):
    return duckdb.sql(sql).df()

In [ ]:
import duckdb

In [ ]:
globals().keys()


dict_keys(['__name__', '__doc__', '__package__', '__loader__', '__spec__', '__builtin__', '__builtins__', '_ih', '_oh', '_dh', 'In', 'Out', 'get_ipython', 'exit', 'quit', '_', '__', '___', '_i', '_ii', '_iii', '_i1', '_i2', '_2', '_i3', 'pd', 'np', '_i4', 'n_leads', 'lead_ids', 'industries', 'company_sizes', 'regions', 'lead_sources', 'experiment_group', 'leads', '_4', '_i5', '_5', '_i6', '_6', '_i7', '_i8', 'email_events', '_i9', 'possible_hours', '_i10', '_i11', '_i12', '_i13', '_13', '_i14', '_14', '_i15', '_15', '_i16', '_i17', '_17', '_i18', '_18', '_i19', '_19', '_i20', '_20', '_i21', '_i22', 'datetime', 'timedelta', '_i23', 'start_date', '_i24', '_i25', '_i26', '_i27', '_i28', 'engagement_base', '_28', '_i29', '_i30', '_i31', '_31', '_i32', '_32', '_i33', '_i34', '_34', '_i35', '_35', '_i36', '_36', '_i37', 'control_open', '_i38', 'treatment_lift', '_i39', 'get_open_prob', '_i40', '_i41', '_i42', 'control_ctr', 'treatment_ctr', '_i43', '_i44', '_i45', '_i46', 'engagement_events'

In [ ]:
q("""
SELECT
    experiment_group,
    COUNT(*) AS leads,
    ROUND(
        COUNT(*) * 100.0 /
        SUM(COUNT(*)) OVER (),
        2
    ) AS pct_of_total
FROM leads
GROUP BY experiment_group
""")

,experiment_group,leads,pct_of_total
0,Treatment,24993,49.99
1,Control,25007,50.01


In [ ]:
q("""
SELECT
    COUNT(*) AS mismatched_records
FROM email_events e
JOIN leads l
    ON e.lead_id = l.lead_id
WHERE e.experiment_group <> l.experiment_group
""")

,mismatched_records
0,0


In [ ]:
q("""
SELECT
    experiment_group,
    COUNT(*) AS emails,
    SUM(delivered) AS delivered_emails,
    ROUND(
        SUM(delivered) * 100.0 /
        COUNT(*),
        2
    ) AS delivery_rate_pct
FROM email_events
GROUP BY experiment_group
""")

,experiment_group,emails,delivered_emails,delivery_rate_pct
0,Treatment,24993,24499.0,98.02
1,Control,25007,24484.0,97.91


In [ ]:
q("""
SELECT
    experiment_group,
    industry,
    COUNT(*) AS leads
FROM leads
GROUP BY
    experiment_group,
    industry
ORDER BY
    industry,
    experiment_group
""")

,experiment_group,industry,leads
0,Control,Finance,6220
1,Treatment,Finance,6221
2,Control,Healthcare,5092
3,Treatment,Healthcare,4834
4,Control,Manufacturing,5028
5,Treatment,Manufacturing,4997
6,Control,Technology,8667
7,Treatment,Technology,8941


In [ ]:
q("""
SELECT
    experiment_group,
    region,
    COUNT(*) AS leads
FROM leads
GROUP BY
    experiment_group,
    region
ORDER BY
    region,
    experiment_group
""")

,experiment_group,region,leads
0,Control,EMEA,5043
1,Treatment,EMEA,5116
2,Control,UK,7417
3,Treatment,UK,7417
4,Control,US,12547
5,Treatment,US,12460


In [ ]:
q("""
SELECT
    experiment_group,
    company_size,
    COUNT(*) AS leads
FROM leads
GROUP BY
    experiment_group,
    company_size
ORDER BY
    company_size,
    experiment_group
""")

,experiment_group,company_size,leads
0,Control,Enterprise,3769
1,Treatment,Enterprise,3661
2,Control,Mid-Market,8907
3,Treatment,Mid-Market,8782
4,Control,SMB,12331
5,Treatment,SMB,12550


In [ ]:
q("""
SELECT
    SUM(CASE WHEN industry IS NULL THEN 1 ELSE 0 END) AS missing_industry,
    SUM(CASE WHEN region IS NULL THEN 1 ELSE 0 END) AS missing_region,
    SUM(CASE WHEN company_size IS NULL THEN 1 ELSE 0 END) AS missing_company_size,
    SUM(CASE WHEN lead_source IS NULL THEN 1 ELSE 0 END) AS missing_lead_source
FROM leads
""")

,missing_industry,missing_region,missing_company_size,missing_lead_source
0,0.0,0.0,0.0,0.0


In [ ]:
q("""
SELECT
    experiment_group,
    ROUND(AVG(send_hour), 2) AS avg_send_hour,
    MIN(send_hour) AS min_hour,
    MAX(send_hour) AS max_hour
FROM email_events
GROUP BY experiment_group
""")

,experiment_group,avg_send_hour,min_hour,max_hour
0,Treatment,11.6,8,16
1,Control,10.0,10,10


In [ ]:
q("""
SELECT
    experiment_group,
    COUNT(DISTINCT lead_id) AS leads,
    COUNT(*) AS emails,
    SUM(delivered) AS delivered,
    ROUND(
        SUM(delivered) * 100.0 / COUNT(*),
        2
    ) AS delivery_rate_pct,
    ROUND(
        AVG(send_hour),
        2
    ) AS avg_send_hour
FROM email_events
GROUP BY experiment_group
""")

,experiment_group,leads,emails,delivered,delivery_rate_pct,avg_send_hour
0,Treatment,24993,24993,24499.0,98.02,11.6
1,Control,25007,25007,24484.0,97.91,10.0


In [ ]:
q("""
SELECT
    COUNT(*) AS emails
FROM email_events
""")

q("""
SELECT
    COUNT(*) AS engagement_rows
FROM engagement_events
""")

q("""
SELECT
    COUNT(*) AS opportunity_rows
FROM opportunities
""")

,opportunity_rows
0,1858


In [ ]:
q("""
SELECT
    e.email_id,
    e.lead_id,
    e.experiment_group,
    e.delivered,

    g.opened,
    g.clicked,
    g.unsubscribed,
    g.complaint,

    o.deal_value,
    o.closed_won

FROM email_events e

LEFT JOIN engagement_events g
    ON e.email_id = g.email_id

LEFT JOIN opportunities o
    ON e.email_id = o.email_id
LIMIT 10
""")

,email_id,lead_id,experiment_group,delivered,opened,clicked,unsubscribed,complaint,deal_value,closed_won
0,90,90,Treatment,True,1.0,0,0,0,18081.0,1
1,175,175,Treatment,True,1.0,0,0,0,8024.0,0
2,178,178,Treatment,False,0.0,0,0,0,2620.0,0
3,191,191,Treatment,True,1.0,0,0,0,847.0,0
4,298,298,Control,True,1.0,0,0,0,24073.0,0
5,324,324,Treatment,True,0.0,1,0,0,23266.0,0
6,328,328,Control,True,0.0,0,0,0,8309.0,1
7,349,349,Control,True,1.0,0,0,0,3902.0,0
8,418,418,Treatment,True,1.0,0,0,0,21764.0,0
9,458,458,Treatment,True,0.0,1,0,0,2497.0,0


In [ ]:
q("""
SELECT
    e.experiment_group,

    COUNT(*) AS emails,

    SUM(e.delivered) AS delivered,

    SUM(g.opened) AS opens,

    SUM(g.clicked) AS clicks,

    SUM(g.unsubscribed) AS unsubscribes,

    SUM(g.complaint) AS complaints,

    ROUND(
        100.0 * SUM(g.opened)
        / SUM(e.delivered),
        2
    ) AS open_rate_pct,

    ROUND(
        100.0 * SUM(g.clicked)
        / SUM(e.delivered),
        2
    ) AS ctr_pct,

    ROUND(
        100.0 * SUM(g.unsubscribed)
        / SUM(e.delivered),
        2
    ) AS unsubscribe_rate_pct

FROM email_events e

LEFT JOIN engagement_events g
    ON e.email_id = g.email_id

GROUP BY e.experiment_group
""")

,experiment_group,emails,delivered,opens,clicks,unsubscribes,complaints,open_rate_pct,ctr_pct,unsubscribe_rate_pct
0,Treatment,24993,24499.0,9263.0,1640.0,97.0,11.0,37.81,6.69,0.40
1,Control,25007,24484.0,8707.0,1553.0,91.0,13.0,35.56,6.34,0.37


In [ ]:
q("""
SELECT
    e.experiment_group,

    COUNT(DISTINCT o.lead_id) AS opportunities,

    SUM(o.deal_value) AS pipeline_value,

    SUM(
        CASE
            WHEN o.closed_won = 1
            THEN o.deal_value
            ELSE 0
        END
    ) AS won_revenue

FROM email_events e

LEFT JOIN opportunities o
    ON e.email_id = o.email_id

GROUP BY e.experiment_group
""")

,experiment_group,opportunities,pipeline_value,won_revenue
0,Control,620,8706557.0,2013222.0
1,Treatment,1238,17313902.0,4395768.0


In [ ]:
q("""
SELECT
    experiment_group,
    COUNT(*) AS opportunity_rows,
    COUNT(DISTINCT lead_id) AS unique_leads,
    SUM(deal_value) AS pipeline_value,
    AVG(deal_value) AS avg_deal_size,
    SUM(CASE WHEN closed_won = 1 THEN deal_value ELSE 0 END) AS won_revenue
FROM opportunities
GROUP BY experiment_group
""")

,experiment_group,opportunity_rows,unique_leads,pipeline_value,avg_deal_size,won_revenue
0,Treatment,1238,1238,17313902.0,13985.381260,4395768.0
1,Control,620,620,8706557.0,14042.833871,2013222.0


In [ ]:
q("""
SELECT
    COUNT(*) AS email_rows
FROM email_events
""")

,email_rows
0,50000


In [ ]:
q("""
SELECT
    COUNT(*) AS opportunity_rows
FROM opportunities
""")

,opportunity_rows
0,1858


In [ ]:
q("""
SELECT
    experiment_group,
    COUNT(*) AS opportunities
FROM opportunities
GROUP BY experiment_group
""")

,experiment_group,opportunities
0,Treatment,1238
1,Control,620


In [ ]:
q("""
SELECT
    l.experiment_group,
    COUNT(*) AS leads,

    COALESCE(o.opportunities,0) AS opportunities,

    ROUND(
        100.0 * COALESCE(o.opportunities,0)
        / COUNT(*),
        2
    ) AS opportunity_rate_pct

FROM leads l

LEFT JOIN (

    SELECT
        experiment_group,
        COUNT(*) AS opportunities
    FROM opportunities
    GROUP BY experiment_group

) o

ON l.experiment_group = o.experiment_group

GROUP BY
    l.experiment_group,
    o.opportunities
""")

,experiment_group,leads,opportunities,opportunity_rate_pct
0,Control,25007,620,2.48
1,Treatment,24993,1238,4.95


In [ ]:
q("""
SELECT
    l.industry,
    l.experiment_group,

    COUNT(*) AS leads,

    COUNT(o.lead_id) AS opportunities,

    ROUND(
        100.0 * COUNT(o.lead_id) / COUNT(*),
        2
    ) AS opportunity_rate_pct,

    ROUND(
        COALESCE(SUM(o.deal_value),0),
        0
    ) AS pipeline_value,

    ROUND(
        COALESCE(SUM(
            CASE
                WHEN o.closed_won = 1
                THEN o.deal_value
                ELSE 0
            END
        ),0),
        0
    ) AS won_revenue

FROM leads l

LEFT JOIN opportunities o
    ON l.lead_id = o.lead_id

GROUP BY
    l.industry,
    l.experiment_group

ORDER BY
    l.industry,
    l.experiment_group
""")

,industry,experiment_group,leads,opportunities,opportunity_rate_pct,pipeline_value,won_revenue
0,Finance,Control,6220,154,2.48,1828990.0,418190.0
1,Finance,Treatment,6221,310,4.98,4396332.0,1042056.0
2,Healthcare,Control,5092,93,1.83,1414564.0,368145.0
3,Healthcare,Treatment,4834,165,3.41,1972570.0,374798.0
4,Manufacturing,Control,5028,73,1.45,893077.0,160954.0
5,Manufacturing,Treatment,4997,59,1.18,752671.0,234235.0
6,Technology,Control,8667,300,3.46,4569926.0,1065933.0
7,Technology,Treatment,8941,704,7.87,10192329.0,2744679.0


In [ ]:
q("""
SELECT
    l.region,
    l.experiment_group,

    COUNT(*) AS leads,

    COUNT(o.lead_id) AS opportunities,

    ROUND(
        100.0 * COUNT(o.lead_id) / COUNT(*),
        2
    ) AS opportunity_rate_pct,

    ROUND(
        COALESCE(SUM(o.deal_value),0),
        0
    ) AS pipeline_value

FROM leads l

LEFT JOIN opportunities o
    ON l.lead_id = o.lead_id

GROUP BY
    l.region,
    l.experiment_group

ORDER BY
    l.region,
    l.experiment_group
""")

,region,experiment_group,leads,opportunities,opportunity_rate_pct,pipeline_value
0,EMEA,Control,5043,112,2.22,1736458.0
1,EMEA,Treatment,5116,262,5.12,3422881.0
2,UK,Control,7417,189,2.55,2364337.0
3,UK,Treatment,7417,358,4.83,5058990.0
4,US,Control,12547,319,2.54,4605762.0
5,US,Treatment,12460,618,4.96,8832031.0


In [ ]:
q("""
SELECT
    l.company_size,
    l.experiment_group,

    COUNT(*) AS leads,

    COUNT(o.lead_id) AS opportunities,

    ROUND(
        100.0 * COUNT(o.lead_id) / COUNT(*),
        2
    ) AS opportunity_rate_pct,

    ROUND(
        COALESCE(SUM(o.deal_value),0),
        0
    ) AS pipeline_value

FROM leads l

LEFT JOIN opportunities o
    ON l.lead_id = o.lead_id

GROUP BY
    l.company_size,
    l.experiment_group

ORDER BY
    l.company_size,
    l.experiment_group
""")

,company_size,experiment_group,leads,opportunities,opportunity_rate_pct,pipeline_value
0,Enterprise,Control,3769,84,2.23,1323044.0
1,Enterprise,Treatment,3661,194,5.30,2873118.0
2,Mid-Market,Control,8907,243,2.73,3290322.0
3,Mid-Market,Treatment,8782,435,4.95,5616483.0
4,SMB,Control,12331,293,2.38,4093191.0
5,SMB,Treatment,12550,609,4.85,8824301.0


In [ ]:
q("""
SELECT
    e.experiment_group,
    g.clicked,
    COUNT(*) AS emails,
    COUNT(o.email_id) AS opportunities,
    ROUND(
        100.0 * COUNT(o.email_id) / COUNT(*),
        2
    ) AS opportunity_rate_pct
FROM email_events e
JOIN engagement_events g
    ON e.email_id = g.email_id
LEFT JOIN opportunities o
    ON e.email_id = o.email_id
GROUP BY
    e.experiment_group,
    g.clicked
ORDER BY
    e.experiment_group,
    g.clicked
""")

,experiment_group,clicked,emails,opportunities,opportunity_rate_pct
0,Control,0,23454,534,2.28
1,Control,1,1553,86,5.54
2,Treatment,0,23353,1077,4.61
3,Treatment,1,1640,161,9.82


In [ ]:
q("""
SELECT
    g.clicked,
    COUNT(*) AS emails,
    COUNT(o.email_id) AS opportunities,
    ROUND(
        100.0 * COUNT(o.email_id) / COUNT(*),
        2
    ) AS opportunity_rate_pct
FROM engagement_events g
LEFT JOIN opportunities o
    ON g.email_id = o.email_id
GROUP BY g.clicked
ORDER BY g.clicked
""")

,clicked,emails,opportunities,opportunity_rate_pct
0,0,46807,1611,3.44
1,1,3193,247,7.74


In [ ]:
q("""
SELECT
    e.experiment_group,
    g.clicked,
    COUNT(*) AS emails,
    COUNT(o.email_id) AS opportunities,
    ROUND(
        100.0 * COUNT(o.email_id) / COUNT(*),
        2
    ) AS opportunity_rate_pct
FROM email_events e
JOIN engagement_events g
    ON e.email_id = g.email_id
LEFT JOIN opportunities o
    ON e.email_id = o.email_id
GROUP BY
    e.experiment_group,
    g.clicked
ORDER BY
    e.experiment_group,
    g.clicked
""")

,experiment_group,clicked,emails,opportunities,opportunity_rate_pct
0,Control,0,23454,534,2.28
1,Control,1,1553,86,5.54
2,Treatment,0,23353,1077,4.61
3,Treatment,1,1640,161,9.82


In [ ]:
q("""
WITH funnel AS (
SELECT
    l.experiment_group,
    COUNT(DISTINCT l.lead_id) AS leads,

    COUNT(DISTINCT CASE
        WHEN e.opened = 1
        THEN l.lead_id END) AS openers,

    COUNT(DISTINCT CASE
        WHEN e.clicked = 1
        THEN l.lead_id END) AS clickers,

    COUNT(DISTINCT CASE
        WHEN o.lead_id IS NOT NULL
        THEN l.lead_id END) AS opps

FROM leads l
LEFT JOIN email_events em
    ON l.lead_id = em.lead_id
LEFT JOIN engagement_events e
    ON em.email_id = e.email_id
LEFT JOIN opportunities o
    ON l.lead_id = o.lead_id

GROUP BY 1
)

SELECT *
FROM funnel
""")

,experiment_group,leads,openers,clickers,opps
0,Control,25007,8707,1553,620
1,Treatment,24993,9263,1640,1238


In [ ]:
q("""
WITH lead_level AS (

SELECT
    l.lead_id,
    l.experiment_group,

    MAX(COALESCE(e.opened,0)) AS opened,

    MAX(
        CASE
            WHEN o.lead_id IS NOT NULL
            THEN 1
            ELSE 0
        END
    ) AS opp

FROM leads l

LEFT JOIN email_events em
    ON l.lead_id = em.lead_id

LEFT JOIN engagement_events e
    ON em.email_id = e.email_id

LEFT JOIN opportunities o
    ON l.lead_id = o.lead_id

GROUP BY 1,2
)

SELECT
    experiment_group,
    opened,

    COUNT(*) AS leads,

    SUM(opp) AS opps,

    ROUND(
        SUM(opp)*100.0/COUNT(*),
        2
    ) AS opp_rate

FROM lead_level

GROUP BY 1,2
ORDER BY 1,2
""")

,experiment_group,opened,leads,opps,opp_rate
0,Control,0.0,16300,366.0,2.25
1,Control,1.0,8707,254.0,2.92
2,Treatment,0.0,15730,655.0,4.16
3,Treatment,1.0,9263,583.0,6.29


In [ ]:
engagement_base["engagement_score"] = (
    engagement_base["opened"].fillna(0) * 1
    +
    engagement_base["clicked"] * 3
)

In [ ]:
base_mqo = {
    "Technology": 0.010,
    "Finance": 0.008,
    "Healthcare": 0.006,
    "Manufacturing": 0.004
}

In [ ]:
def get_opportunity_prob(row):

    base = base_mqo[row["industry"]]

    score_multiplier = {
        0: 1.0,
        1: 2.0,
        3: 5.0,
        4: 8.0
    }

    return base * score_multiplier[
        row["engagement_score"]
    ]

In [ ]:
engagement_base["engagement_score"].value_counts().sort_index()

,count
engagement_score,
0.0,29984
1.0,16823
3.0,2046
4.0,1147


In [ ]:
engagement_base.columns.tolist()

['email_id',
 'lead_id',
 'experiment_group',
 'send_date',
 'send_datetime',
 'send_hour',
 'delivered',
 'industry',
 'company_size',
 'region',
 'open_prob',
 'opened',
 'clicked',
 'unsubscribed',
 'complaint',
 'bot_open',
 'tracking_error',
 'true_opened',
 'engagement_score']

In [ ]:
engagement_base["engagement_score"] = (
    engagement_base["opened"].fillna(0) * 1
    +
    engagement_base["clicked"] * 3
)

In [ ]:
engagement_base["engagement_score"].value_counts().sort_index()

,count
engagement_score,
0.0,29984
1.0,16823
3.0,2046
4.0,1147


In [ ]:
op_base = engagement_base.copy()

In [ ]:
op_base.columns.tolist()

['email_id',
 'lead_id',
 'experiment_group',
 'send_date',
 'send_datetime',
 'send_hour',
 'delivered',
 'industry',
 'company_size',
 'region',
 'open_prob',
 'opened',
 'clicked',
 'unsubscribed',
 'complaint',
 'bot_open',
 'tracking_error',
 'true_opened',
 'engagement_score']

In [ ]:
base_mqo = {
    "Technology": 0.010,
    "Finance": 0.008,
    "Healthcare": 0.006,
    "Manufacturing": 0.004
}

In [ ]:
def get_opportunity_prob(row):

    base = base_mqo[row["industry"]]

    score_multiplier = {
        0: 1.0,
        1: 2.0,
        3: 5.0,
        4: 8.0
    }

    return base * score_multiplier[
        row["engagement_score"]
    ]

In [ ]:
op_base["opportunity_prob"] = op_base.apply(
    get_opportunity_prob,
    axis=1
)

In [ ]:
op_base["opportunity_created"] = np.random.binomial(
    1,
    op_base["opportunity_prob"]
)

In [ ]:
op_base.groupby(
    "engagement_score"
)["opportunity_created"].mean()

,opportunity_created
engagement_score,
0.0,0.007471
1.0,0.014801
3.0,0.038123
4.0,0.053182


In [ ]:
op_base.groupby(
    ["engagement_score", "experiment_group"]
)["opportunity_created"].mean()

engagement_score  experiment_group
0.0               Control             0.007520
                  Treatment           0.007420
1.0               Control             0.015317
                  Treatment           0.014315
3.0               Control             0.045680
                  Treatment           0.030799
4.0               Control             0.047619
                  Treatment           0.058236
Name: opportunity_created, dtype: float64

In [ ]:
opportunities = op_base[
    op_base["opportunity_created"] == 1
].copy()

In [ ]:
opportunities["deal_value"] = np.random.lognormal(
    mean=9,
    sigma=1,
    size=len(opportunities)
)

In [ ]:
opportunities["deal_value"] = opportunities[
    "deal_value"
].clip(
    lower=1000,
    upper=250000
)

In [ ]:
opportunities["deal_value"] = (
    opportunities["deal_value"]
    .round(0)
)

In [ ]:
opportunities["closed_won"] = np.random.binomial(
    1,
    0.25,
    len(opportunities)
)

In [ ]:
op_base.groupby(
    "experiment_group"
)["opportunity_created"].mean()

,opportunity_created
experiment_group,
Control,0.012477
Treatment,0.012003


In [ ]:
opportunities.groupby(
    "experiment_group"
)["deal_value"].sum()

,deal_value
experiment_group,
Control,4472026.0
Treatment,4193170.0


In [ ]:
opportunities.groupby(
    "experiment_group"
)["closed_won"].mean()

,closed_won
experiment_group,
Control,0.246795
Treatment,0.270000


In [ ]:
opportunities.groupby(
    "experiment_group"
)["deal_value"].mean()

,deal_value
experiment_group,
Control,14333.416667
Treatment,13977.233333


In [ ]:
deal_value = np.random.lognormal(
    mean=9.5,
    sigma=0.9
)

In [ ]:
q("""
SELECT
    l.experiment_group,

    COUNT(DISTINCT l.lead_id) AS leads,

    COUNT(DISTINCT CASE
        WHEN e.opened = 1
        THEN l.lead_id
    END) AS openers,

    COUNT(DISTINCT CASE
        WHEN e.clicked = 1
        THEN l.lead_id
    END) AS clickers,

    COUNT(DISTINCT o.lead_id) AS opportunities

FROM leads l

LEFT JOIN email_events em
    ON l.lead_id = em.lead_id

LEFT JOIN engagement_base e
    ON em.email_id = e.email_id

LEFT JOIN opportunities o
    ON l.lead_id = o.lead_id

GROUP BY l.experiment_group
ORDER BY l.experiment_group
""")

,experiment_group,leads,openers,clickers,opportunities
0,Control,25007,8707,1553,312
1,Treatment,24993,9263,1640,300


In [ ]:
q("""
SELECT
    experiment_group,
    engagement_score,

    COUNT(*) AS leads

FROM engagement_base

GROUP BY 1,2
ORDER BY 1,2
""")

,experiment_group,engagement_score,leads
0,Control,0.0,15293
1,Control,1.0,8161
2,Control,3.0,1007
3,Control,4.0,546
4,Treatment,0.0,14691
5,Treatment,1.0,8662
6,Treatment,3.0,1039
7,Treatment,4.0,601


In [ ]:
q("""
SELECT
    e.experiment_group,
    e.engagement_score,

    COUNT(*) AS leads,

    COUNT(o.lead_id) AS opps,

    ROUND(
        COUNT(o.lead_id)*100.0/
        COUNT(*),
        2
    ) AS opp_rate

FROM engagement_base e

LEFT JOIN opportunities o
    ON e.lead_id = o.lead_id

GROUP BY 1,2
ORDER BY 1,2
""")

,experiment_group,engagement_score,leads,opps,opp_rate
0,Control,0.0,15293,115,0.75
1,Control,1.0,8161,125,1.53
2,Control,3.0,1007,46,4.57
3,Control,4.0,546,26,4.76
4,Treatment,0.0,14691,109,0.74
5,Treatment,1.0,8662,124,1.43
6,Treatment,3.0,1039,32,3.08
7,Treatment,4.0,601,35,5.82


In [ ]:
q("""
SELECT
    experiment_group,
    AVG(engagement_score) AS avg_engagement_score,
    MEDIAN(engagement_score) AS median_engagement_score
FROM engagement_base
GROUP BY 1
""")

,experiment_group,avg_engagement_score,median_engagement_score
0,Control,0.534490,0.0
1,Treatment,0.567479,0.0


In [ ]:
q("""
SELECT
    engagement_score,
    experiment_group,
    COUNT(*) AS leads
FROM engagement_base
GROUP BY 1,2
ORDER BY 1,2
""")

,engagement_score,experiment_group,leads
0,0.0,Control,15293
1,0.0,Treatment,14691
2,1.0,Control,8161
3,1.0,Treatment,8662
4,3.0,Control,1007
5,3.0,Treatment,1039
6,4.0,Control,546
7,4.0,Treatment,601


In [ ]:
q("""
SELECT
    experiment_group,
    COUNT(*) AS opps,
    SUM(deal_value) AS pipeline
FROM opportunities
GROUP BY 1
""")

,experiment_group,opps,pipeline
0,Treatment,300,4193170.0
1,Control,312,4472026.0


In [ ]:
q("""
SELECT
    experiment_group,

    COUNT(*) FILTER (
        WHERE closed_won = 1
    ) AS won_deals,

    SUM(
        CASE
            WHEN closed_won = 1
            THEN deal_value
        END
    ) AS revenue

FROM opportunities

GROUP BY 1
""")

,experiment_group,won_deals,revenue
0,Treatment,81,907841.0
1,Control,77,1065900.0


In [ ]:
q("""
SELECT
    e.engagement_score,

    COUNT(*) AS opps,

    AVG(o.deal_value) AS avg_deal_size,

    SUM(o.deal_value) AS pipeline

FROM opportunities o

JOIN engagement_base e
    ON o.lead_id = e.lead_id

GROUP BY 1
ORDER BY 1
""")

,engagement_score,opps,avg_deal_size,pipeline
0,0.0,224,14596.705357,3269662.0
1,1.0,249,13232.036145,3294777.0
2,3.0,78,14253.551282,1111777.0
3,4.0,61,16212.786885,988980.0
